In [38]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [39]:
patients = pd.read_csv('../data/processed/patients_clean.csv', parse_dates=['registration_date'])
vitals = pd.read_csv('../data/processed/vital_signs_clean.csv', parse_dates=['timestamp'])
history = pd.read_csv('../data/processed/clinical_history_clean.csv')
labs = pd.read_csv('../data/processed/laboratory_results_clean.csv', parse_dates=['timestamp'])
outcomes = pd.read_csv('../data/processed/sepsis_outcomes_clean.csv', parse_dates=['diagnosis_time'])

tables = {'patients': patients, 'vitals': vitals, 'history': history, 'labs': labs, 'outcomes': outcomes}
for name, df in tables.items():
    print(f"{name:10s} shape: ]={df.shape}")

patients   shape: ]=(5000, 5)
vitals     shape: ]=(99737, 8)
history    shape: ]=(12459, 6)
labs       shape: ]=(20034, 8)
outcomes   shape: ]=(5000, 6)


In [40]:
patients.columns

Index(['patient_id', 'age', 'gender', 'medical_conditions',
       'registration_date'],
      dtype='str')

In [41]:
vitals.columns

Index(['observation_id', 'patient_id', 'timestamp', 'heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure'],
      dtype='str')

In [42]:
labs.columns

Index(['lab_id', 'patient_id', 'timestamp', 'white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count'],
      dtype='str')

In [43]:
vital_cols = [
    'heart_rate', 
    'temperature', 
    'oxygen_saturation', 
    'respiratory_rate', 
    'blood_pressure'
]

labs_cols = [
    'white_cell_count', 
    'crp',
    'lactate', 
    'creatinine', 
    'platelet_count'
]

In [44]:
outcomes.columns

Index(['outcome_id', 'patient_id', 'sepsis_event', 'diagnosis_time',
       'hospitalisation_required', 'outcome_status'],
      dtype='str')

In [45]:
# prepare a random number generator for reproducibility 
rng = np.random.default_rng(7)

# returns a random prediction time for a patient based on their vitals data and sepsis outcome
def get_prediction_time(row, vitals_df):
    if row['sepsis_event']:
        return row['diagnosis_time'] - pd.Timedelta(hours=9)
    pv = vitals_df[vitals_df['patient_id'] == row['patient_id']]
    start, end = pv['timestamp'].min(), pv['timestamp'].max()
    span_hours = max((end - start).total_seconds() / 3600, 1)
    offset = rng.uniform(0.4, 0.9) * span_hours
    return start + pd.Timedelta(hours=offset)


outcomes = outcomes.copy()
outcomes['prediction_time'] = outcomes.apply(get_prediction_time, vitals_df=vitals, axis=1)
outcomes[['patient_id', 'sepsis_event', 'diagnosis_time', 'prediction_time']].head()


,patient_id,sepsis_event,diagnosis_time,prediction_time
0,1,True,2024-12-13 05:36:00,2024-12-12 20:36:00.000000000
1,2,False,NaT,2025-08-28 06:26:55.786260611
2,3,True,2024-07-19 13:24:00,2024-07-19 04:24:00.000000000
3,4,True,2024-08-21 21:07:00,2024-08-21 12:07:00.000000000
4,5,False,NaT,2024-04-11 14:53:44.568198471


In [46]:
outcomes[['patient_id', 'sepsis_event', 'diagnosis_time', 'prediction_time']].head(15)

,patient_id,sepsis_event,diagnosis_time,prediction_time
0,1,True,2024-12-13 05:36:00,2024-12-12 20:36:00.000000000
1,2,False,NaT,2025-08-28 06:26:55.786260611
2,3,True,2024-07-19 13:24:00,2024-07-19 04:24:00.000000000
3,4,True,2024-08-21 21:07:00,2024-08-21 12:07:00.000000000
4,5,False,NaT,2024-04-11 14:53:44.568198471
5,6,False,NaT,2024-06-10 07:48:58.256127454
6,7,True,2024-01-29 10:48:00,2024-01-29 01:48:00.000000000
7,8,False,NaT,2025-08-15 03:04:57.752682750
8,9,False,NaT,2025-09-11 12:46:43.210235189
9,10,True,2024-11-02 06:47:00,2024-11-01 21:47:00.000000000


In [47]:
LOOKBACK_HOURS = 6

def vital_features(pid, cutoff, df):
    window = df[
        (df['patient_id'] == pid) &
        (df['timestamp'] <= cutoff) &
        (df['timestamp'] >= cutoff - pd.Timedelta(hours=LOOKBACK_HOURS))
    ]
    if window.empty:
        window = df[(df['patient_id'] == pid) & (df['timestamp'] <= cutoff)].tail(1)
    feats = {}
    for col in vital_cols:
        vals = window[col]
        feats[f'{col}_mean'] = vals.mean()
        feats[f'{col}_min'] = vals.min()
        feats[f'{col}_max'] = vals.max()
        feats[f'{col}_std'] = vals.std() if len(vals) > 1 else 0.0
        feats[f'{col}_last'] = vals.iloc[-1] if len(vals) > 1 else np.nan
        
        if len(window) > 1:
            hours = (window['timestamp'].iloc[-1] - window['timestamp'].iloc[0]).total_seconds() / 3600
            feats[f'{col}_rate_per_hr'] = (vals.iloc[-1] - vals.iloc[0]) / hours if hours > 0 else 0.0
        else:
            feats[f'{col}_rate_per_hr'] = 0.0  
    
    return feats  
           
          

In [48]:
vital_feature_rows = [
    {'patient_id': pid, **vital_features(pid, cutoff, vitals)}
    for pid, cutoff in zip(outcomes['patient_id'], outcomes['prediction_time'])
]
vital_features_df = pd.DataFrame(vital_feature_rows)
vital_features_df.head(10)

,patient_id,heart_rate_mean,heart_rate_min,heart_rate_max,heart_rate_std,heart_rate_last,heart_rate_rate_per_hr,temperature_mean,temperature_min,temperature_max,temperature_std,temperature_last,temperature_rate_per_hr,oxygen_saturation_mean,oxygen_saturation_min,oxygen_saturation_max,oxygen_saturation_std,oxygen_saturation_last,oxygen_saturation_rate_per_hr,respiratory_rate_mean,respiratory_rate_min,respiratory_rate_max,respiratory_rate_std,respiratory_rate_last,respiratory_rate_rate_per_hr,blood_pressure_mean,blood_pressure_min,blood_pressure_max,blood_pressure_std,blood_pressure_last,blood_pressure_rate_per_hr
0,1,78.200000,78.2,78.2,0.000000,NaN,0.000000,36.750000,36.75,36.75,0.000000,NaN,0.000000,97.000000,97.0,97.0,0.000000,NaN,0.000000,16.700000,16.7,16.7,0.000000,NaN,0.000000,114.300000,114.3,114.3,0.000000,NaN,0.000000
1,2,77.100000,77.1,77.1,0.000000,NaN,0.000000,36.720000,36.72,36.72,0.000000,NaN,0.000000,97.500000,97.5,97.5,0.000000,NaN,0.000000,13.800000,13.8,13.8,0.000000,NaN,0.000000,99.100000,99.1,99.1,0.000000,NaN,0.000000
2,3,86.900000,86.9,86.9,0.000000,NaN,0.000000,37.540000,37.54,37.54,0.000000,NaN,0.000000,95.900000,95.9,95.9,0.000000,NaN,0.000000,15.300000,15.3,15.3,0.000000,NaN,0.000000,114.800000,114.8,114.8,0.000000,NaN,0.000000
3,4,85.400000,85.4,85.4,0.000000,NaN,0.000000,37.700000,37.70,37.70,0.000000,NaN,0.000000,96.800000,96.8,96.8,0.000000,NaN,0.000000,17.600000,17.6,17.6,0.000000,NaN,0.000000,115.900000,115.9,115.9,0.000000,NaN,0.000000
4,5,78.300000,78.3,78.3,0.000000,NaN,0.000000,36.970000,36.97,36.97,0.000000,NaN,0.000000,98.100000,98.1,98.1,0.000000,NaN,0.000000,13.800000,13.8,13.8,0.000000,NaN,0.000000,118.300000,118.3,118.3,0.000000,NaN,0.000000
5,6,69.900000,65.7,74.1,5.939697,74.1,13.263158,36.285000,36.03,36.54,0.360624,36.54,0.805263,97.450000,96.3,98.6,1.626346,98.6,3.631579,18.250000,17.4,19.1,1.202082,17.4,-2.684211,133.050000,132.5,133.6,0.777817,133.6,1.736842
6,7,72.900000,72.9,72.9,0.000000,NaN,0.000000,37.220000,37.22,37.22,0.000000,NaN,0.000000,95.500000,95.5,95.5,0.000000,NaN,0.000000,10.300000,10.3,10.3,0.000000,NaN,0.000000,110.200000,110.2,110.2,0.000000,NaN,0.000000
7,8,84.366667,78.9,87.1,4.734272,78.9,-3.727273,37.183333,37.17,37.19,0.011547,37.17,-0.009091,95.333333,94.5,96.6,1.115049,96.6,0.954545,15.566667,14.8,16.0,0.665833,16.0,0.045455,124.933333,120.5,127.9,3.911948,126.4,-0.681818
8,9,84.800000,84.8,84.8,0.000000,NaN,0.000000,36.310000,36.31,36.31,0.000000,NaN,0.000000,97.500000,97.5,97.5,0.000000,NaN,0.000000,16.400000,16.4,16.4,0.000000,NaN,0.000000,128.600000,128.6,128.6,0.000000,NaN,0.000000
9,10,71.800000,71.8,71.8,0.000000,NaN,0.000000,37.270000,37.27,37.27,0.000000,NaN,0.000000,98.500000,98.5,98.5,0.000000,NaN,0.000000,17.300000,17.3,17.3,0.000000,NaN,0.000000,133.800000,133.8,133.8,0.000000,NaN,0.000000


In [49]:
LAB_LOOKBACK_HOURS = 24

def lab_features(pid, cutoff, df):
    window = df[
        (df['patient_id'] == pid) &
        (df['timestamp'] <= cutoff) &
        (df['timestamp'] >= cutoff - pd.Timedelta(hours=LAB_LOOKBACK_HOURS))
    ]
    if window.empty:
        window = df[(df['patient_id'] == pid) & (df['timestamp'] <= cutoff)].tail(1)
    feats = {}
    for col in labs_cols:
        vals = window[col]
        feats[f'{col}_mean'] = vals.mean()
        feats[f'{col}_last'] = vals.iloc[-1] if len(vals) > 1 else np.nan
        
    return feats

In [50]:
lab_feature_rows = [
    {'patient_id': pid, **lab_features(pid, cutoff, labs)}
    for pid, cutoff in zip(outcomes['patient_id'], outcomes['prediction_time'])
]
lab_features_df = pd.DataFrame(lab_feature_rows)
lab_features_df.head(10)

,patient_id,white_cell_count_mean,white_cell_count_last,crp_mean,crp_last,lactate_mean,lactate_last,creatinine_mean,creatinine_last,platelet_count_mean,platelet_count_last
0,1,8.370000,8.49,12.950000,20.3,1.565,1.60,1.085000,1.09,220.500000,233.0
1,2,8.110000,NaN,2.700000,NaN,0.550,NaN,1.020000,NaN,339.000000,NaN
2,3,7.093333,6.88,39.533333,4.9,1.350,1.27,0.886667,0.93,247.333333,235.0
3,4,10.335000,10.94,4.500000,5.4,0.925,1.09,1.020000,1.01,316.000000,316.0
4,5,7.420000,NaN,9.400000,NaN,1.140,NaN,0.820000,NaN,243.000000,NaN
5,6,10.640000,NaN,6.700000,NaN,0.890,NaN,0.670000,NaN,290.000000,NaN
6,7,8.042500,8.56,5.800000,6.7,0.320,0.33,0.692500,0.54,329.500000,331.0
7,8,6.905000,7.68,1.850000,3.0,0.680,0.68,0.545000,0.65,304.500000,302.0
8,9,7.100000,NaN,5.900000,NaN,0.570,NaN,1.300000,NaN,265.000000,NaN
9,10,5.900000,NaN,8.300000,NaN,1.010,NaN,1.010000,NaN,291.000000,NaN


In [51]:
patients.columns

Index(['patient_id', 'age', 'gender', 'medical_conditions',
       'registration_date'],
      dtype='str')

In [52]:
patients['gender'].unique()

<ArrowStringArray>
['Male', 'Female', 'Other/Not specified']
Length: 3, dtype: str

In [53]:
static  = patients[['patient_id', 'age', 'gender']].copy()

static['comorbidity_count'] = patients['medical_conditions'].apply(
    lambda x: 0 if pd.isna(x) or x == 'None reported'
    else len(x.split(','))
)

static = pd.get_dummies(static, columns=['gender'], drop_first=True)
static.head()

,patient_id,age,comorbidity_count,gender_Male,gender_Other/Not specified
0,1,66,0,True,False
1,2,42,0,True,False
2,3,74,0,False,False
3,4,77,0,False,False
4,5,25,1,True,False


In [54]:
feature = (
    static
    .merge(vital_features_df, on='patient_id')
    .merge(lab_features_df, on='patient_id')
    .merge(outcomes[['patient_id', 'sepsis_event']], on='patient_id')
)

feature_cols = [
    c for c in feature.columns
    if c not in ('patient_id', 'sepsis_event')
]

numeric_cols = feature[feature_cols].select_dtypes(include='number').columns

feature[numeric_cols] = feature[numeric_cols].fillna(
    feature[numeric_cols].median()
)

feature['sepsis_event'] = feature['sepsis_event'].astype(int)

print(feature.shape)

(5000, 46)


In [55]:
feature.head(10)

,patient_id,age,comorbidity_count,gender_Male,gender_Other/Not specified,heart_rate_mean,heart_rate_min,heart_rate_max,heart_rate_std,heart_rate_last,heart_rate_rate_per_hr,temperature_mean,temperature_min,temperature_max,temperature_std,temperature_last,temperature_rate_per_hr,oxygen_saturation_mean,oxygen_saturation_min,oxygen_saturation_max,oxygen_saturation_std,oxygen_saturation_last,oxygen_saturation_rate_per_hr,respiratory_rate_mean,respiratory_rate_min,respiratory_rate_max,respiratory_rate_std,respiratory_rate_last,respiratory_rate_rate_per_hr,blood_pressure_mean,blood_pressure_min,blood_pressure_max,blood_pressure_std,blood_pressure_last,blood_pressure_rate_per_hr,white_cell_count_mean,white_cell_count_last,crp_mean,crp_last,lactate_mean,lactate_last,creatinine_mean,creatinine_last,platelet_count_mean,platelet_count_last,sepsis_event
0,1,66,0,True,False,78.200000,78.2,78.2,0.000000,80.2,0.000000,36.750000,36.75,36.75,0.000000,36.85,0.000000,97.000000,97.0,97.0,0.000000,97.1,0.000000,16.700000,16.7,16.7,0.000000,16.4,0.000000,114.300000,114.3,114.3,0.000000,120.5,0.000000,8.370000,8.49,12.950000,20.3,1.565,1.60,1.085000,1.09,220.500000,233.0,1
1,2,42,0,True,False,77.100000,77.1,77.1,0.000000,80.2,0.000000,36.720000,36.72,36.72,0.000000,36.85,0.000000,97.500000,97.5,97.5,0.000000,97.1,0.000000,13.800000,13.8,13.8,0.000000,16.4,0.000000,99.100000,99.1,99.1,0.000000,120.5,0.000000,8.110000,7.59,2.700000,6.8,0.550,1.02,1.020000,0.92,339.000000,259.0,0
2,3,74,0,False,False,86.900000,86.9,86.9,0.000000,80.2,0.000000,37.540000,37.54,37.54,0.000000,36.85,0.000000,95.900000,95.9,95.9,0.000000,97.1,0.000000,15.300000,15.3,15.3,0.000000,16.4,0.000000,114.800000,114.8,114.8,0.000000,120.5,0.000000,7.093333,6.88,39.533333,4.9,1.350,1.27,0.886667,0.93,247.333333,235.0,1
3,4,77,0,False,False,85.400000,85.4,85.4,0.000000,80.2,0.000000,37.700000,37.70,37.70,0.000000,36.85,0.000000,96.800000,96.8,96.8,0.000000,97.1,0.000000,17.600000,17.6,17.6,0.000000,16.4,0.000000,115.900000,115.9,115.9,0.000000,120.5,0.000000,10.335000,10.94,4.500000,5.4,0.925,1.09,1.020000,1.01,316.000000,316.0,1
4,5,25,1,True,False,78.300000,78.3,78.3,0.000000,80.2,0.000000,36.970000,36.97,36.97,0.000000,36.85,0.000000,98.100000,98.1,98.1,0.000000,97.1,0.000000,13.800000,13.8,13.8,0.000000,16.4,0.000000,118.300000,118.3,118.3,0.000000,120.5,0.000000,7.420000,7.59,9.400000,6.8,1.140,1.02,0.820000,0.92,243.000000,259.0,0
5,6,37,3,True,False,69.900000,65.7,74.1,5.939697,74.1,13.263158,36.285000,36.03,36.54,0.360624,36.54,0.805263,97.450000,96.3,98.6,1.626346,98.6,3.631579,18.250000,17.4,19.1,1.202082,17.4,-2.684211,133.050000,132.5,133.6,0.777817,133.6,1.736842,10.640000,7.59,6.700000,6.8,0.890,1.02,0.670000,0.92,290.000000,259.0,0
6,7,63,2,True,False,72.900000,72.9,72.9,0.000000,80.2,0.000000,37.220000,37.22,37.22,0.000000,36.85,0.000000,95.500000,95.5,95.5,0.000000,97.1,0.000000,10.300000,10.3,10.3,0.000000,16.4,0.000000,110.200000,110.2,110.2,0.000000,120.5,0.000000,8.042500,8.56,5.800000,6.7,0.320,0.33,0.692500,0.54,329.500000,331.0,1
7,8,55,2,False,False,84.366667,78.9,87.1,4.734272,78.9,-3.727273,37.183333,37.17,37.19,0.011547,37.17,-0.009091,95.333333,94.5,96.6,1.115049,96.6,0.954545,15.566667,14.8,16.0,0.665833,16.0,0.045455,124.933333,120.5,127.9,3.911948,126.4,-0.681818,6.905000,7.68,1.850000,3.0,0.680,0.68,0.545000,0.65,304.500000,302.0,0
8,9,60,0,False,True,84.800000,84.8,84.8,0.000000,80.2,0.000000,36.310000,36.31,36.31,0.000000,36.85,0.000000,97.500000,97.5,97.5,0.000000,97.1,0.000000,16.400000,16.4,16.4,0.000000,16.4,0.000000,128.600000,128.6,128.6,0.000000,120.5,0.000000,7.100000,7.59,5.900000,6.8,0.570,1.02,1.300000,0.92,265.000000,259.0,0
9,10,45,2,True,False,71.800000,71.8,71.8,0.000000,80.2,0.000000,37.270000,37.27,37.27,0.000000,36.85,0.000000,98.500000,98.5,98.5,0.000000,97.1,0.000000,17.300000,17.3,17.3,0.000000,16.4,0.000000,133.800000,133.8,133.8,0.000000,120.5,0.000000,5.900000,7.59,8.300000,6.8,1.010,1.02,1.010000,0.92,291.000000,259.0,1


In [56]:
feature.columns

Index(['patient_id', 'age', 'comorbidity_count', 'gender_Male',
       'gender_Other/Not specified', 'heart_rate_mean', 'heart_rate_min',
       'heart_rate_max', 'heart_rate_std', 'heart_rate_last',
       'heart_rate_rate_per_hr', 'temperature_mean', 'temperature_min',
       'temperature_max', 'temperature_std', 'temperature_last',
       'temperature_rate_per_hr', 'oxygen_saturation_mean',
       'oxygen_saturation_min', 'oxygen_saturation_max',
       'oxygen_saturation_std', 'oxygen_saturation_last',
       'oxygen_saturation_rate_per_hr', 'respiratory_rate_mean',
       'respiratory_rate_min', 'respiratory_rate_max', 'respiratory_rate_std',
       'respiratory_rate_last', 'respiratory_rate_rate_per_hr',
       'blood_pressure_mean', 'blood_pressure_min', 'blood_pressure_max',
       'blood_pressure_std', 'blood_pressure_last',
       'blood_pressure_rate_per_hr', 'white_cell_count_mean',
       'white_cell_count_last', 'crp_mean', 'crp_last', 'lactate_mean',
       'lactate_last

In [57]:
check_cols = ['heart_rate_last', 'oxygen_saturation_last', 'crp_last', 'lactate_last']
feature.groupby('sepsis_event')[check_cols].mean()

,heart_rate_last,oxygen_saturation_last,crp_last,lactate_last
sepsis_event,,,,
0,79.343296,97.252519,6.643296,1.010233
1,81.418522,96.947826,8.230522,1.049004


In [58]:
corr = feature[feature_cols + ['sepsis_event']].corr()['sepsis_event'].drop('sepsis_event')
corr.sort_values(key=abs, ascending=False).head(10)

age                       0.669143
comorbidity_count         0.250626
oxygen_saturation_last   -0.180073
crp_last                  0.174321
heart_rate_last           0.163888
respiratory_rate_last     0.161481
crp_mean                  0.148235
oxygen_saturation_min    -0.143772
oxygen_saturation_mean   -0.123134
heart_rate_max            0.120548
Name: sepsis_event, dtype: float64

In [59]:
feature.to_csv('../data/processed/sepsis_features.csv', index=False)